# Session Midterm: Dot Product & Matrix Transpose with CUDA

**Sinh viên:** Nguyễn Thanh Trọng  
**MSSV:** B2305615  
**Môn học:** CT225H - Parallel Programming with CUDA

---

## Mô tả bài toán

### Bài toán 1: Tích vô hướng (Dot Product)
Cho hai vector $A$ và $B$ có cùng kích thước $N$. Tính: $Result = \sum_{i=0}^{N-1} A_i \times B_i$

Sử dụng mẫu thuật toán **Reduction** trên GPU với shared memory.

### Bài toán 2: Chuyển vị ma trận (Matrix Transpose)
Cho ma trận $A$ kích thước $N \times M$. Tạo ma trận chuyển vị $A^T$ kích thước $M \times N$ sao cho $A^T[j][i] = A[i][j]$.

Sử dụng shared memory tiling để tối ưu hóa truy cập bộ nhớ (memory coalescing).

In [4]:
!nvidia-smi

Tue Mar  3 02:59:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             19W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bài toán 1: Tích vô hướng (Dot Product)

### Thuật toán
1. Mỗi thread tính tích `A[i] * B[i]` và lưu vào shared memory
2. Thực hiện **parallel reduction** trong shared memory (cộng dồn theo cây nhị phân)
3. Block sum được ghi ra global memory
4. CPU cộng các block sums lại thành kết quả cuối cùng

### DAG Reduction
```
Level 0:  a0*b0   a1*b1   a2*b2   a3*b3   a4*b4   a5*b5   a6*b6   a7*b7
Level 1:  sum01           sum23           sum45           sum67
Level 2:  sum0123                         sum4567
Level 3:  total_sum
```

In [5]:
%%writefile dot_product.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N 1000000
#define BLOCK_SIZE 256

// ===================== CPU Sequential =====================
double dotProductCPU(float *A, float *B, int n) {
    double sum = 0.0;
    for (int i = 0; i < n; i++) {
        sum += (double)A[i] * B[i];
    }
    return sum;
}

// ===================== CUDA Kernel: Dot Product with Reduction =====================
__global__ void dotProductKernel(float *A, float *B, double *blockSums, int n) {
    __shared__ double sdata[BLOCK_SIZE];

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Each thread computes its element-wise product
    sdata[tid] = (idx < n) ? (double)A[idx] * B[idx] : 0.0;
    __syncthreads();

    // Parallel reduction in shared memory
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) {
            sdata[tid] += sdata[tid + s];
        }
        __syncthreads();
    }

    // Write block result
    if (tid == 0) {
        blockSums[blockIdx.x] = sdata[0];
    }
}

int main() {
    printf("=== DOT PRODUCT (N = %d) ===\n\n", N);

    // Allocate host memory
    float *h_A = (float*)malloc(N * sizeof(float));
    float *h_B = (float*)malloc(N * sizeof(float));

    // Initialize vectors
    srand(42);
    for (int i = 0; i < N; i++) {
        h_A[i] = (float)(rand() % 100) / 100.0f;
        h_B[i] = (float)(rand() % 100) / 100.0f;
    }

    // ---- CPU Sequential ----
    clock_t cpu_start = clock();
    double cpuResult = dotProductCPU(h_A, h_B, N);
    clock_t cpu_end = clock();
    double cpuTime = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;
    printf("[CPU Sequential]  Result: %.6f  Time: %.4f ms\n", cpuResult, cpuTime);

    // ---- GPU CUDA ----
    float *d_A, *d_B;
    int numBlocks = (N + BLOCK_SIZE - 1) / BLOCK_SIZE;
    double *d_blockSums;
    double *h_blockSums = (double*)malloc(numBlocks * sizeof(double));

    cudaMalloc(&d_A, N * sizeof(float));
    cudaMalloc(&d_B, N * sizeof(float));
    cudaMalloc(&d_blockSums, numBlocks * sizeof(double));

    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(float), cudaMemcpyHostToDevice);

    // Timing with CUDA events
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    dotProductKernel<<<numBlocks, BLOCK_SIZE>>>(d_A, d_B, d_blockSums, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTimeMs = 0;
    cudaEventElapsedTime(&gpuTimeMs, start, stop);

    cudaMemcpy(h_blockSums, d_blockSums, numBlocks * sizeof(double), cudaMemcpyDeviceToHost);

    // Final reduction on CPU
    double gpuResult = 0.0;
    for (int i = 0; i < numBlocks; i++) {
        gpuResult += h_blockSums[i];
    }

    printf("[GPU CUDA]        Result: %.6f  Time: %.4f ms\n", gpuResult, gpuTimeMs);

    // Verification & Speedup
    double diff = (cpuResult - gpuResult);
    if (diff < 0) diff = -diff;
    int correct = (diff < 1e-2);
    double speedup = cpuTime / gpuTimeMs;

    printf("\n--- Verification ---\n");
    printf("Difference: %.10f\n", diff);
    printf("Correct: %s\n", correct ? "Yes" : "No");
    printf("Speedup (CPU/GPU): %.2fx\n", speedup);

    // Cleanup
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_blockSums);
    free(h_A);
    free(h_B);
    free(h_blockSums);

    return 0;
}


Writing dot_product.cu


In [6]:
!nvcc -o dot_product dot_product.cu -arch=sm_75 && ./dot_product

=== DOT PRODUCT (N = 1000000) ===

[CPU Sequential]  Result: 244219.001864  Time: 2.9270 ms
[GPU CUDA]        Result: 244219.001864  Time: 0.3111 ms

--- Verification ---
Difference: 0.0000000153
Correct: Yes
Speedup (CPU/GPU): 9.41x


## Bài toán 2: Chuyển vị Ma trận (Matrix Transpose)

### Thuật toán
1. **Naive kernel:** Mỗi thread đọc `A[row][col]` và ghi vào `B[col][row]` — đơn giản nhưng không tối ưu bộ nhớ
2. **Shared memory tiled kernel:**
   - Đọc tile từ A vào shared memory (coalesced read)
   - `__syncthreads()`
   - Ghi tile đã chuyển vị từ shared memory ra B (coalesced write)
   - Tránh bank conflicts bằng padding (`TILE_DIM+1`)

In [7]:
%%writefile matrix_transpose.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define ROWS 1024
#define COLS 2048
#define TILE_DIM 32
#define BLOCK_ROWS 8

// ===================== CPU Sequential =====================
void transposeCPU(float *in, float *out, int rows, int cols) {
    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) {
            out[j * rows + i] = in[i * cols + j];
        }
    }
}

// ===================== CUDA Naive Transpose =====================
__global__ void transposeNaive(float *in, float *out, int rows, int cols) {
    int x = blockIdx.x * TILE_DIM + threadIdx.x;
    int y = blockIdx.y * TILE_DIM + threadIdx.y;

    for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS) {
        if ((y + j) < rows && x < cols) {
            out[x * rows + (y + j)] = in[(y + j) * cols + x];
        }
    }
}

// ===================== CUDA Shared Memory Transpose =====================
__global__ void transposeShared(float *in, float *out, int rows, int cols) {
    __shared__ float tile[TILE_DIM][TILE_DIM + 1]; // +1 to avoid bank conflicts

    int x = blockIdx.x * TILE_DIM + threadIdx.x;
    int y = blockIdx.y * TILE_DIM + threadIdx.y;

    // Load tile from input (coalesced read)
    for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS) {
        if ((y + j) < rows && x < cols) {
            tile[threadIdx.y + j][threadIdx.x] = in[(y + j) * cols + x];
        }
    }
    __syncthreads();

    // Write transposed tile to output (coalesced write)
    x = blockIdx.y * TILE_DIM + threadIdx.x;  // swap block indices
    y = blockIdx.x * TILE_DIM + threadIdx.y;

    for (int j = 0; j < TILE_DIM; j += BLOCK_ROWS) {
        if ((y + j) < cols && x < rows) {
            out[(y + j) * rows + x] = tile[threadIdx.x][threadIdx.y + j];
        }
    }
}

int verify(float *ref, float *result, int size) {
    for (int i = 0; i < size; i++) {
        if (ref[i] != result[i]) return 0;
    }
    return 1;
}

int main() {
    printf("=== MATRIX TRANSPOSE (%d x %d) ===\n\n", ROWS, COLS);

    int sizeIn = ROWS * COLS;
    int sizeOut = COLS * ROWS;
    size_t bytesIn = sizeIn * sizeof(float);
    size_t bytesOut = sizeOut * sizeof(float);

    // Allocate host memory
    float *h_A   = (float*)malloc(bytesIn);
    float *h_cpu  = (float*)malloc(bytesOut);
    float *h_naive = (float*)malloc(bytesOut);
    float *h_shared = (float*)malloc(bytesOut);

    // Initialize matrix
    srand(42);
    for (int i = 0; i < sizeIn; i++) {
        h_A[i] = (float)(rand() % 1000) / 100.0f;
    }

    // ---- CPU Sequential ----
    clock_t cpu_start = clock();
    transposeCPU(h_A, h_cpu, ROWS, COLS);
    clock_t cpu_end = clock();
    double cpuTime = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;
    printf("[CPU Sequential]    Time: %.4f ms\n", cpuTime);

    // ---- GPU Setup ----
    float *d_A, *d_out;
    cudaMalloc(&d_A, bytesIn);
    cudaMalloc(&d_out, bytesOut);
    cudaMemcpy(d_A, h_A, bytesIn, cudaMemcpyHostToDevice);

    dim3 blockDim(TILE_DIM, BLOCK_ROWS);
    dim3 gridDim((COLS + TILE_DIM - 1) / TILE_DIM, (ROWS + TILE_DIM - 1) / TILE_DIM);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float gpuTime;

    // ---- GPU Naive ----
    cudaMemset(d_out, 0, bytesOut);
    cudaEventRecord(start);
    transposeNaive<<<gridDim, blockDim>>>(d_A, d_out, ROWS, COLS);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(h_naive, d_out, bytesOut, cudaMemcpyDeviceToHost);
    printf("[GPU Naive]         Time: %.4f ms  Correct: %s\n", gpuTime,
           verify(h_cpu, h_naive, sizeOut) ? "Yes" : "No");
    double naiveTime = gpuTime;

    // ---- GPU Shared Memory ----
    cudaMemset(d_out, 0, bytesOut);
    cudaEventRecord(start);
    transposeShared<<<gridDim, blockDim>>>(d_A, d_out, ROWS, COLS);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(h_shared, d_out, bytesOut, cudaMemcpyDeviceToHost);
    printf("[GPU Shared Memory] Time: %.4f ms  Correct: %s\n", gpuTime,
           verify(h_cpu, h_shared, sizeOut) ? "Yes" : "No");
    double sharedTime = gpuTime;

    // ---- Speedup ----
    printf("\n--- Speedup ---\n");
    printf("CPU / Naive:  %.2fx\n", cpuTime / naiveTime);
    printf("CPU / Shared: %.2fx\n", cpuTime / sharedTime);
    printf("Naive / Shared: %.2fx\n", naiveTime / sharedTime);

    // Cleanup
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_A);
    cudaFree(d_out);
    free(h_A);
    free(h_cpu);
    free(h_naive);
    free(h_shared);

    return 0;
}


Overwriting matrix_transpose.cu


In [8]:
!nvcc -o matrix_transpose matrix_transpose.cu -arch=sm_75 && ./matrix_transpose

=== MATRIX TRANSPOSE (1024 x 2048) ===

[CPU Sequential]    Time: 40.5120 ms
[GPU Naive]         Time: 0.4028 ms  Correct: Yes
[GPU Shared Memory] Time: 0.0951 ms  Correct: Yes

--- Speedup ---
CPU / Naive:  100.58x
CPU / Shared: 426.12x
Naive / Shared: 4.24x
